# 🧠 Scratch TextCNN - Smart MCQ Solver (With Advanced Wandb Logging)

### 📌 Overview & Wandb Tracking Features
This notebook implements a **scratch TextCNN classifier** trained with **5-fold GroupKFold cross-validation** and **rich Epoch-Wise, Fold-Wise, and Model-Wise Weights & Biases (Wandb) Logging**.

- **Primary Kaggle Metric**: `MAP@3` (Mean Average Precision at 3)
- **Secondary Evaluation Metrics**:
  - `Top-1 Accuracy`: standard classification accuracy
  - `Top-3 Accuracy`: recall — ground truth within top 3
  - `F1 / Precision / Recall`: macro-averaged binary metrics on the pairwise `label`
- **Architecture**: `Embedding → parallel Conv1d (kernels 3, 4, 5) → global max-pool → Linear head`
- **Tokenizer**: `bert-base-uncased` WordPiece (~30k tokens)
- **CV**: 5-fold `GroupKFold` by `qid` — all 5 options of one question stay in the same fold
- **Test predictions**: averaged across the 5 fold models

In [1]:
# Cell 1: Core Imports & Dependencies
import os
import re
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, log_loss
from transformers import AutoTokenizer

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Core libraries imported successfully! device={device}")

Core libraries imported successfully! device=cuda


In [2]:
# Cell 2: Weights & Biases (Wandb) Authentication & Setup
try:
    import wandb
    WANDB_ENTITY  = "23f2001849-dl-genai-project"
    WANDB_PROJECT = "23f2001849-t22026"

    key = None
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception:
        key = os.environ.get("WANDB_API_KEY")

    wandb.login(key=key, relogin=True)
    USE_WANDB = True
    print("Wandb initialized successfully!")
except Exception as e:
    USE_WANDB = False
    print(f"Wandb running in offline fallback mode ({e})")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Wandb initialized successfully!


In [3]:
# Cell 3: Dataset Loading & Setup
def load_datasets():
    train_candidates = [
        "train.csv",
        "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv",
        "/kaggle/input/smart-mcq-solver-challenge/train.csv"
    ]
    test_candidates = [
        "test.csv",
        "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv",
        "/kaggle/input/smart-mcq-solver-challenge/test.csv"
    ]
    train_path = next((p for p in train_candidates if os.path.exists(p)), train_candidates[0])
    test_path  = next((p for p in test_candidates  if os.path.exists(p)), test_candidates[0])
    return pd.read_csv(train_path), pd.read_csv(test_path)

train, test = load_datasets()
OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
print(f"Train shape: {train.shape} | Test shape: {test.shape}")

Train shape: (2000, 8) | Test shape: (500, 7)


In [4]:
# Cell 4: Comprehensive Metric Evaluation Functions
def apk(actual, predicted, k=3):
    if actual in predicted[:k]:
        return 1.0 / (predicted[:k].index(actual) + 1.0)
    return 0.0

def mapk(actuals, preds, k=3):
    return float(np.mean([apk(a, p, k) for a, p in zip(actuals, preds)]))

def compute_all_metrics(actuals, pred_lists):
    map3_score = mapk(actuals, pred_lists)
    top1_preds = [p[0] for p in pred_lists]
    top1_acc   = accuracy_score(actuals, top1_preds)
    top3_acc   = float(np.mean([a in p[:3] for a, p in zip(actuals, pred_lists)]))
    p, r, f1, _ = precision_recall_fscore_support(actuals, top1_preds, average='macro', zero_division=0)
    return {'map3': map3_score, 'top1_accuracy': top1_acc, 'top3_accuracy': top3_acc,
            'precision': p, 'recall': r, 'f1_score': f1}

def rank_preds(pw, scores, orig_df):
    tmp = pw.copy()
    tmp['__s__'] = scores
    out = []
    for qid in orig_df['id']:
        q = tmp[tmp['qid'] == qid].sort_values('__s__', ascending=False)
        out.append(q['option'].tolist()[:3])
    return out

In [5]:
# Cell 5: Pairwise DataFrame (text = prompt [SEP] option, label = 1 if correct)
def make_pairwise_dataset(df, is_train=True):
    rows = []
    for _, r in df.iterrows():
        qid = r['id']
        prompt = str(r['prompt'])
        ans = r.get('answer', None) if is_train else None
        for c in OPTION_COLS:
            rows.append({
                'qid':    qid,
                'option': c,
                'text':   prompt + ' [SEP] ' + str(r[c]),
                'label':  int(is_train and ans == c),
            })
    return pd.DataFrame(rows)

pw_train = make_pairwise_dataset(train, is_train=True)
pw_test  = make_pairwise_dataset(test,  is_train=False)
print(f"pw_train: {pw_train.shape} | pw_test: {pw_test.shape}")

pw_train: (10000, 4) | pw_test: (2500, 4)


In [6]:
# Cell 6: Tokenize with bert-base-uncased AutoTokenizer -> TensorDatasets
TOKENIZER_NAME = 'bert-base-uncased'
MAX_LEN        = 128
BATCH_SIZE     = 32

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

def encode(texts):
    enc = tokenizer(list(texts), max_length=MAX_LEN, truncation=True,
                    padding='max_length', return_tensors='pt')
    return enc['input_ids'], enc['attention_mask']

train_ids, train_mask = encode(pw_train['text'])
test_ids,  test_mask  = encode(pw_test['text'])
labels = torch.tensor(pw_train['label'].values, dtype=torch.float32)
y_all  = pw_train['label'].values

test_loader = DataLoader(TensorDataset(test_ids, test_mask),
                         batch_size=BATCH_SIZE, shuffle=False)
print(f"vocab_size: {tokenizer.vocab_size} | train_ids: {tuple(train_ids.shape)} | test_ids: {tuple(test_ids.shape)}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

vocab_size: 30522 | train_ids: (10000, 128) | test_ids: (2500, 128)


In [7]:
# Cell 7: TextCNN Architecture + Train/Predict Helpers
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, n_filters=64, kernels=(3, 4, 5)):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=tokenizer.pad_token_id)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, n_filters, k, padding=k // 2) for k in kernels])
        self.fc = nn.Linear(n_filters * len(kernels), 1)

    def forward(self, input_ids, attention_mask=None):
        emb = self.embedding(input_ids).transpose(1, 2)               # (B, E, L)
        pooled = [torch.relu(c(emb)).max(dim=2).values for c in self.convs]
        return self.fc(torch.cat(pooled, dim=1)).squeeze(1)

def train_epoch(model, loader, optim, loss_fn):
    model.train()
    tot, n = 0.0, 0
    for ids, mask, y in loader:
        ids, mask, y = ids.to(device), mask.to(device), y.to(device)
        optim.zero_grad()
        loss = loss_fn(model(ids, mask), y)
        loss.backward()
        optim.step()
        tot += loss.item() * y.size(0); n += y.size(0)
    return tot / max(n, 1)

def predict(model, loader):
    model.eval()
    scores = []
    with torch.no_grad():
        for batch in loader:
            ids, mask = batch[0].to(device), batch[1].to(device)
            scores.extend(torch.sigmoid(model(ids, mask)).cpu().numpy())
    return np.array(scores)

In [8]:
# Cell 8: 5-Fold Training Loop with Epoch-Wise, Fold-Wise & Overall Wandb Logging
gkf    = GroupKFold(n_splits=5)
groups = pw_train['qid'].values

EMBED_DIM = 128
N_FILTERS = 64
KERNELS   = (3, 4, 5)
LR        = 1e-3
EPOCHS    = 5

oof_cnn = np.zeros(len(pw_train))
tst_cnn = np.zeros(len(pw_test))

config_dict = {
    "architecture": "Scratch TextCNN (Embedding -> Conv1d x3 -> MaxPool -> Linear)",
    "dataset": "Smart MCQ Solver Challenge",
    "tokenizer": TOKENIZER_NAME,
    "max_len": MAX_LEN, "batch_size": BATCH_SIZE,
    "embed_dim": EMBED_DIM, "n_filters": N_FILTERS, "kernels": list(KERNELS),
    "lr": LR, "epochs": EPOCHS, "n_splits": 5,
}

if USE_WANDB:
    run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        name="scratch-textcnn-detailed-fold-logs",
        config=config_dict
    )

print("\nStarting 5-Fold Cross Validation...")
fold_metrics_list = []
loss_fn = nn.BCEWithLogitsLoss()

for fold, (ti, vi) in enumerate(gkf.split(pw_train, y_all, groups)):
    print(f"\n--- Fold {fold+1}/5 ---")

    tr_loader = DataLoader(TensorDataset(train_ids[ti], train_mask[ti], labels[ti]),
                           batch_size=BATCH_SIZE, shuffle=True)
    va_loader = DataLoader(TensorDataset(train_ids[vi], train_mask[vi], labels[vi]),
                           batch_size=BATCH_SIZE, shuffle=False)

    model = TextCNN(tokenizer.vocab_size, EMBED_DIM, N_FILTERS, KERNELS).to(device)
    optim = torch.optim.Adam(model.parameters(), lr=LR)

    # Per-epoch training + val loss logging
    for epoch in range(EPOCHS):
        tr_loss = train_epoch(model, tr_loader, optim, loss_fn)
        va_probs = predict(model, va_loader)
        va_loss  = log_loss(y_all[vi], np.clip(va_probs, 1e-6, 1 - 1e-6))
        print(f"  Epoch {epoch+1}/{EPOCHS} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f}")
        if USE_WANDB:
            wandb.log({
                f"fold_{fold+1}/epoch": epoch + 1,
                f"fold_{fold+1}/train_loss": tr_loss,
                f"fold_{fold+1}/val_loss":   va_loss,
            })

    # Final fold predictions + test contribution
    val_scores = predict(model, va_loader)
    oof_cnn[vi] = val_scores
    tst_cnn    += predict(model, test_loader) / 5.0

    # Fold ranking metrics
    val_pw_df   = pw_train.iloc[vi].reset_index(drop=True)
    val_qids    = val_pw_df['qid'].unique()
    val_df_orig = train[train['id'].isin(val_qids)].reset_index(drop=True)
    fold_preds   = rank_preds(val_pw_df, val_scores, val_df_orig)
    fold_actuals = val_df_orig['answer'].tolist()

    m = compute_all_metrics(fold_actuals, fold_preds)
    fold_val_loss = log_loss(y_all[vi], np.clip(val_scores, 1e-6, 1 - 1e-6))
    print(f"  Fold {fold+1} val_loss={fold_val_loss:.4f} | MAP@3={m['map3']:.4f} "
          f"| Acc={m['top1_accuracy']:.4f} | F1={m['f1_score']:.4f}")
    fold_metrics_list.append(m)

    if USE_WANDB:
        wandb.log({
            "fold": fold + 1,
            "fold/val_loss":          fold_val_loss,
            "fold/val_map3":          m['map3'],
            "fold/val_top1_accuracy": m['top1_accuracy'],
            "fold/val_top3_accuracy": m['top3_accuracy'],
            "fold/val_f1_score":      m['f1_score'],
            "fold/val_precision":     m['precision'],
            "fold/val_recall":        m['recall'],
        })

# ----- Overall Out-Of-Fold Performance -----
overall_actuals = train['answer'].tolist()
m_cnn = compute_all_metrics(overall_actuals, rank_preds(pw_train, oof_cnn, train))
oof_loss = log_loss(y_all, np.clip(oof_cnn, 1e-6, 1 - 1e-6))

print("\n==================================================")
print("       OVERALL OUT-OF-FOLD PERFORMANCE SUMMARY")
print("==================================================")
print(f" Primary Kaggle Metric (MAP@3): {m_cnn['map3']:.4f}")
print(f" Top-1 Accuracy               : {m_cnn['top1_accuracy']:.4f}")
print(f" Top-3 Recall Accuracy        : {m_cnn['top3_accuracy']:.4f}")
print(f" Macro F1 Score               : {m_cnn['f1_score']:.4f}")
print(f" Macro Precision              : {m_cnn['precision']:.4f}")
print(f" Macro Recall                 : {m_cnn['recall']:.4f}")
print(f" Pairwise Log Loss            : {oof_loss:.4f}")
print("==================================================\n")

if USE_WANDB:
    wandb.log({
        "oof/overall_map3":          m_cnn['map3'],
        "oof/overall_top1_accuracy": m_cnn['top1_accuracy'],
        "oof/overall_top3_accuracy": m_cnn['top3_accuracy'],
        "oof/overall_f1_score":      m_cnn['f1_score'],
        "oof/overall_precision":     m_cnn['precision'],
        "oof/overall_recall":        m_cnn['recall'],
        "oof/overall_val_loss":      oof_loss,
    })
    table = wandb.Table(columns=["Model Architecture", "MAP@3 Score", "Top-1 Accuracy",
                                  "Top-3 Accuracy", "F1 Score", "Precision", "Recall"])
    table.add_data("Scratch TextCNN (5-fold avg)",
                   m_cnn['map3'], m_cnn['top1_accuracy'], m_cnn['top3_accuracy'],
                   m_cnn['f1_score'], m_cnn['precision'], m_cnn['recall'])
    wandb.log({"model_comparative_summary": table})
    wandb.finish()


Starting 5-Fold Cross Validation...

--- Fold 1/5 ---
  Epoch 1/5 | train_loss=0.4023 | val_loss=0.2691
  Epoch 2/5 | train_loss=0.1963 | val_loss=0.1474
  Epoch 3/5 | train_loss=0.1005 | val_loss=0.0912
  Epoch 4/5 | train_loss=0.0567 | val_loss=0.0507
  Epoch 5/5 | train_loss=0.0397 | val_loss=0.0501
  Fold 1 val_loss=0.0501 | MAP@3=1.0000 | Acc=1.0000 | F1=1.0000

--- Fold 2/5 ---
  Epoch 1/5 | train_loss=0.4106 | val_loss=0.2650
  Epoch 2/5 | train_loss=0.2019 | val_loss=0.1305
  Epoch 3/5 | train_loss=0.1022 | val_loss=0.0899
  Epoch 4/5 | train_loss=0.0630 | val_loss=0.0569
  Epoch 5/5 | train_loss=0.0382 | val_loss=0.0390
  Fold 2 val_loss=0.0390 | MAP@3=1.0000 | Acc=1.0000 | F1=1.0000

--- Fold 3/5 ---
  Epoch 1/5 | train_loss=0.4085 | val_loss=0.3117
  Epoch 2/5 | train_loss=0.2025 | val_loss=0.1468
  Epoch 3/5 | train_loss=0.1013 | val_loss=0.1036
  Epoch 4/5 | train_loss=0.0614 | val_loss=0.0544
  Epoch 5/5 | train_loss=0.0411 | val_loss=0.0445
  Fold 3 val_loss=0.0445 | MA

fold,▁▃▅▆█
fold/val_f1_score,▁▁▁▁▁
fold/val_loss,█▁▄▃▂
fold/val_map3,▁▁▁▁▁
fold/val_precision,▁▁▁▁▁
fold/val_recall,▁▁▁▁▁
fold/val_top1_accuracy,▁▁▁▁▁
fold/val_top3_accuracy,▁▁▁▁▁
fold_1/epoch,▁▃▅▆█
fold_1/train_loss,█▄▂▁▁
+20,...


In [9]:
# Cell 9: Final Submission Generation & Comparative Report Table
test_preds = rank_preds(pw_test, tst_cnn, test)

submission = pd.DataFrame({
    'ID': test['id'],
    'Prediction': [' '.join(p) for p in test_preds]
})
submission.to_csv('submission.csv', index=False)
print("Successfully generated submission.csv!")

assert len(submission) == len(test), f"Expected {len(test)} rows, got {len(submission)}"
assert submission['Prediction'].str.match(r'^[A-E] [A-E] [A-E]$').all(), "Invalid prediction format!"

# Comparative Summary Markdown Table for Project Report
report_df = pd.DataFrame([{"Model": "Scratch TextCNN (5-fold avg)", **m_cnn}]).set_index("Model")
print("\n=== FINAL MODEL COMPARATIVE REPORT ===")
print(report_df.round(4).to_markdown())

Successfully generated submission.csv!

=== FINAL MODEL COMPARATIVE REPORT ===
| Model                        |   map3 |   top1_accuracy |   top3_accuracy |   precision |   recall |   f1_score |
|:-----------------------------|-------:|----------------:|----------------:|------------:|---------:|-----------:|
| Scratch TextCNN (5-fold avg) |      1 |               1 |               1 |           1 |        1 |          1 |
